# Phaseless AFQMC with Restricted CISD Trial: Code Walkthrough

This notebook explains how **phaseless auxiliary-field quantum Monte Carlo (AFQMC)** with a **restricted CISD trial wavefunction** is implemented in `ad_afqmc_prototype`.

## Table of Contents
1. [Overall Code Structure](#1-overall-code-structure)
2. [Key Data Structures](#2-key-data-structures)
3. [Trial Wavefunction & Overlap](#3-trial-wavefunction--overlap)
4. [Propagation (Trotter Decomposition)](#4-propagation-trotter-decomposition)
5. [Phaseless Weight Update](#5-phaseless-weight-update)
6. [Force Bias (Importance Sampling)](#6-force-bias-importance-sampling)
7. [Energy Evaluation (Mixed Estimator)](#7-energy-evaluation-mixed-estimator)
8. [Walker Management: Orthogonalization & Stochastic Reconfiguration](#8-walker-management-orthogonalization--stochastic-reconfiguration)
9. [Block-level Loop & Overall Driver](#9-block-level-loop--overall-driver)
10. [Call Graph Summary](#10-call-graph-summary)

---
## 1. Overall Code Structure

```
ad_afqmc_prototype/
├── afqmc.py            # User-facing Afqmc class (.kernel())
├── driver.py           # run_qmc_energy() — equilibration + sampling loop
├── walkers.py          # Walker init, orthogonalization, stochastic reconfiguration
├── staging.py          # PySCF → staged inputs (Cholesky integrals, CI coefficients)
├── stat_utils.py       # Blocking analysis, jackknife error bars
│
├── core/
│   ├── system.py       # System dataclass (norb, nelec, walker_kind)
│   ├── ops.py          # TrialOps / MeasOps protocol containers
│   └── typing.py       # Type aliases
│
├── ham/
│   └── chol.py         # HamChol: h0, h1, chol (Cholesky factors of ERIs)
│
├── trial/
│   └── cisd.py         # CisdTrial, overlap_r, get_rdm1
│
├── meas/
│   └── cisd.py         # CisdMeasCtx, force_bias_kernel_rw_rh, energy_kernel_rw_rh
│
└── prop/
    ├── afqmc.py        # init_prop_state, afqmc_step (phaseless weight update)
    ├── chol_afqmc_ops.py  # _build_prop_ctx, _apply_trotter_r, TrotterOps
    ├── blocks.py       # block() — one QMC block (n_prop_steps + measurement + SR)
    ├── types.py        # PropState, QmcParams
    └── utils.py        # taylor_expm_action (Taylor series for exp)
```

The library is written entirely in **JAX**, so all kernels are JIT-compiled and can run on GPU/TPU.
Functions operating on individual walkers are `vmap`-ed (via `wk.vmap_chunked`) to process the full walker batch.

---
## 2. Key Data Structures

### 2.1 `HamChol` — Hamiltonian  
**File:** `ham/chol.py`

```python
@dataclass(frozen=True)
class HamChol:
    h0:   scalar          # nuclear repulsion + constant
    h1:   (norb, norb)    # one-body integrals (already includes mean-field subtraction)
    chol: (n_chol, norb, norb)  # Cholesky factors L_gamma of ERIs: (ij|kl) ≈ Σ_γ L_γ,ij L_γ,kl
    basis: str            # "restricted" or "generalized"
```

The two-electron repulsion integrals (ERIs) are expressed as a Cholesky decomposition:
$$\langle ij | kl \rangle \approx \sum_{\gamma} L_{\gamma,ij}\, L_{\gamma,kl}$$
This reduces the $O(N^4)$ ERI tensor to an $O(N_\gamma \times N^2)$ matrix, which is the core approximation enabling efficient AFQMC.

---
### 2.2 `CisdTrial` — Trial Wavefunction  
**File:** `trial/cisd.py`, lines 13–51

```python
@dataclass(frozen=True)
class CisdTrial:
    ci1: (nocc, nvir)              # singles: c_{ia}
    ci2: (nocc, nvir, nocc, nvir)  # doubles: c_{iajb}
```

The trial wavefunction is:
$$|\Psi_T\rangle = \left(1 + \sum_{ia} c_{ia}\, \hat{a}^\dagger_a \hat{a}_i + \sum_{iajb} c_{iajb}\, \hat{a}^\dagger_a \hat{a}_i \hat{a}^\dagger_b \hat{a}_j \right)|\Phi_0\rangle$$
where $|\Phi_0\rangle$ is the RHF reference (first `nocc` MOs occupied).
The coefficients `ci1`, `ci2` are typically obtained from a CCSD or CISD calculation.

---
### 2.3 `PropState` — QMC State  
**File:** `prop/types.py`

```python
class PropState(NamedTuple):
    walkers:               (n_walkers, norb, nocc)  # complex Slater determinants
    weights:               (n_walkers,)              # real, positive walker weights
    overlaps:              (n_walkers,)              # <Ψ_T|φ_k> for each walker
    rng_key:               JAX PRNGKey
    pop_control_ene_shift: scalar                    # energy shift for population control
    e_estimate:            scalar                    # EMA of recent energies
    node_encounters:       int                       # count of walkers killed by phaseless
```

Each **walker** $|\phi_k\rangle$ is a complex $(N_{orb} \times N_{occ})$ matrix representing a Slater determinant. The first `nocc` rows form the occupied block.

---
### 2.4 `CholAfqmcCtx` — Propagation Context  
**File:** `prop/chol_afqmc_ops.py`, lines 16–50

```python
@dataclass(frozen=True)
class CholAfqmcCtx:
    dt:          scalar            # time step
    sqrt_dt:     scalar            # √dt
    exp_h1_half: (norb, norb)      # exp(-dt/2 * h1_eff): pre-computed one-body propagator
    mf_shifts:   (n_chol,)         # mean-field shifts: i Tr[L_γ ρ_0]
    h0_prop:     scalar            # -h0 - 0.5 Σ_γ mf_shift_γ²
    chol_flat:   (n_chol, norb²)   # flattened Cholesky for fast VHS construction
    norb:        int
```

---
### 2.5 `CisdMeasCtx` — Measurement Context  
**File:** `meas/cisd.py`, lines 43–59

```python
@dataclass(frozen=True)
class CisdMeasCtx:
    rot_chol: (n_chol, nocc, norb)  # L_γ[:nocc, :]: Cholesky rotated by occupied rows
    lci1:     (n_chol, norb, nocc)  # Σ_t L_γ,it * ci1_pt: precontracted for singles
    cfg:      CisdMeasCfg           # dtype / memory-mode settings
```

These are precomputed once and reused every step to save computation.

---
## 3. Trial Wavefunction & Overlap

**File:** `trial/cisd.py`, function `overlap_r`, lines 62–76

The overlap between a restricted walker $|\phi\rangle$ (an $N_{orb}\times N_{occ}$ Slater determinant matrix $W$) and the CISD trial is:
$$\langle \Psi_T | \phi \rangle = \left(1 + 2 O_1 + O_2\right) \cdot (\det W_{occ})^2$$

where:
- $W_{occ} = W[:N_{occ}, :]$ is the occupied block
- The **Green's function** $G = (W_{occ}^T)^{-1} W^T$ has shape $(N_{occ}, N_{orb})$
- $x = G[:, N_{occ}:]$ are the particle-hole overlaps of shape $(N_{occ}, N_{vir})$
- **Singles overlap:** $O_1 = \sum_{ia} c_{ia}\, x_{ia}$
- **Doubles overlap:** $O_2 = 2\sum_{iajb} c_{iajb}\, x_{ia}\, x_{jb} - \sum_{iajb} c_{iajb}\, x_{ib}\, x_{ja}$

```python
# trial/cisd.py:62-76
def overlap_r(walker, trial_data):
    wocc = walker[:nocc, :]                          # (nocc, nocc)
    green = jnp.linalg.solve(wocc.T, walker.T)       # (nocc, norb)  ← Green's function G
    det0  = jnp.linalg.det(wocc)
    o0    = det0 * det0                              # det² for spin-paired restricted case

    x  = green[:, nocc:]                             # (nocc, nvir)  ← particle-hole sector
    o1 = einsum("ia,ia->", ci1, x)
    o2 = 2.0*einsum("iajb,ia,jb->", ci2, x, x) - einsum("iajb,ib,ja->", ci2, x, x)

    return (1.0 + 2.0*o1 + o2) * o0
```

> **Note:** The factor of 2 in `2.0*o1` arises because both α and β spins contribute equally in the restricted (spin-paired) case.

---
## 4. Propagation (Trotter Decomposition)

### 4.1 Building the Propagation Context
**File:** `prop/chol_afqmc_ops.py`, function `_build_prop_ctx`, lines 103–127

Before the simulation starts, the mean-field shifts and the one-body propagator are precomputed:

$$\bar{x}_\gamma = i \text{Tr}[L_\gamma \rho_0]  \qquad \text{(mean-field shifts)}$$

$$h_1^{\text{eff}} = h_1 - \frac{1}{2}\sum_\gamma L_\gamma^2 - i \sum_\gamma \text{Re}(\bar{x}_\gamma)\, L_\gamma$$

$$U_1 = e^{-\frac{\Delta\tau}{2} h_1^{\text{eff}}}  \qquad \text{(half one-body propagator)}$$

```python
# prop/chol_afqmc_ops.py:103-127
def _build_prop_ctx(ham_data, rdm1, dt, ...):
    mf  = 1j * einsum("gij,ji->g", chol, dm)    # mean-field shifts (n_chol,)
    h1_eff = h1 - 0.5*einsum("gik,gkj->ij", chol, chol) - 1j*einsum("g,gik->ik", real(mf), chol)
    exp_h1_half = expm(-0.5 * dt * h1_eff)       # (norb, norb)
    chol_flat   = chol.reshape(n_chol, norb*norb) # flattened for fast VHS
    return CholAfqmcCtx(dt, sqrt(dt), exp_h1_half, mf, h0_prop, chol_flat, norb)
```

---
### 4.2 Symmetric Trotter Step
**File:** `prop/chol_afqmc_ops.py`, function `_apply_trotter_r`, lines 198–208

Each time step propagates a walker by:
$$|\phi_{\tau+\Delta\tau}\rangle = U_1 \cdot e^{i\sqrt{\Delta\tau}\, V(x)} \cdot U_1\, |\phi_\tau\rangle$$

where $V(x) = \sum_\gamma x_\gamma L_\gamma$ is the **auxiliary-field Hamiltonian** (also called VHS), and $x_\gamma$ are the sampled auxiliary fields.

```python
# prop/chol_afqmc_ops.py:198-208
def _apply_trotter_r(w, field, prop_ctx, n_terms, make_vhs):
    w1 = exp_h1_half @ w                                  # half one-body
    w2 = _apply_two_body_array(w1, field, prop_ctx, ...)  # two-body via VHS
    return exp_h1_half @ w2                               # half one-body again
```

---
### 4.3 Two-body propagator via Taylor expansion
**File:** `prop/utils.py`, function `taylor_expm_action`, lines 20–29  
**File:** `prop/chol_afqmc_ops.py`, function `_make_vhs_split_flat`, lines 83–87

The two-body exponential $e^{i\sqrt{\Delta\tau}\, V}$ is applied via a truncated Taylor series (default `n_terms=6`) **without forming the full matrix exponential**:

$$e^{aM} |v\rangle \approx |v\rangle + aM|v\rangle + \frac{a^2}{2}M^2|v\rangle + \cdots$$

```python
# prop/utils.py:20-29
def taylor_expm_action(a, mat, vecs, n_terms):
    out = term = vecs
    for k in range(1, n_terms + 1):
        term = (a / k) * (mat @ term)   # recurrence: k-th term = (a/k) * mat * (k-1)-th term
        out  = out + term
    return out
```

The VHS matrix is constructed from the sampled (shifted) auxiliary fields $x_\gamma$:
```python
# prop/chol_afqmc_ops.py:83-87
def _make_vhs_split_flat(chol_flat, x, n):
    v_re = real(x) @ chol_flat   # (norb²,)
    v_im = imag(x) @ chol_flat
    return complex(v_re, v_im).reshape(n, n)
```

---
## 5. Phaseless Weight Update

**File:** `prop/afqmc.py`, function `afqmc_step`, lines 79–144

This is the core AFQMC algorithm. Each step:

### Step 1 — Sample Gaussian auxiliary fields
```python
fields = jax.random.normal(subkey, (n_walkers, n_chol))   # x ~ N(0,1)
```

### Step 2 — Shift fields by force bias (importance sampling)
The **importance-sampled fields** $\bar{x}$ shift the Gaussian distribution toward the trial wavefunction:
$$\bar{x}_\gamma = x_\gamma - \Delta x_\gamma, \qquad \Delta x_\gamma = -\sqrt{\Delta\tau}\,(i\, F_\gamma - \bar{x}_\gamma^{\text{mf}})$$

```python
force_bias   = vmap(fb_kernel)(walkers, ...)             # F_γ = <Ψ_T|L_γ|φ> / <Ψ_T|φ>
field_shifts = -sqrt_dt * (1j * force_bias - mf_shifts)  # Δx
shifted_fields = fields - field_shifts                    # x̄ = x - Δx
```

### Step 3 — Apply Trotter propagation
```python
walkers_new = vmap(apply_trotter)(walkers, shifted_fields, prop_ctx, n_exp_terms)
```

### Step 4 — Compute overlap ratio
```python
overlaps_new = vmap(meas_ops.overlap)(walkers_new, trial_data)
ratio = overlaps_new / state.overlaps    # <Ψ_T|φ'> / <Ψ_T|φ>
```

### Step 5 — Phaseless importance function
The importance function combines the overlap ratio with the Hubbard-Stratonovich transformation weight:
$$\mathcal{W}(x) = e^{-\sqrt{\Delta\tau}\, \bar{x}\cdot\bar{x}^{\text{mf}} + x\cdot\Delta x - \frac{1}{2}\Delta x^2 + \Delta\tau\,(E_s + h_0^{\text{prop}})} \cdot \frac{\langle\Psi_T|\phi'\rangle}{\langle\Psi_T|\phi\rangle}$$

```python
# prop/afqmc.py:112-120
exponent = (-sqrt_dt * shift_term + fb_term
            + dt * (pop_shift + h0_prop))
imp_fun  = exp(exponent) * ratio
```

The **phaseless approximation** projects out the imaginary part by taking the cosine of the phase:
$$w_{k}^{\text{new}} = w_k \cdot |\mathcal{W}| \cdot \max(0,\, \cos\theta), \qquad \theta = \arg\left(e^{-\sqrt{\Delta\tau}\bar{x}\cdot\bar{x}^{\text{mf}}} \frac{\langle\Psi_T|\phi'\rangle}{\langle\Psi_T|\phi\rangle}\right)$$

```python
# prop/afqmc.py:119-129
theta  = angle(exp(-sqrt_dt * shift_term) * ratio)  # phase angle
imp_ph = abs(imp_fun) * cos(theta)                  # phaseless importance
imp_ph = where(imp_ph < w_floor, 0.0, imp_ph)       # kill walkers below floor
imp_ph = where(imp_ph > w_cap,   0.0, imp_ph)       # kill walkers above cap
weights_new = state.weights * imp_ph
```

### Step 6 — Population control energy shift
```python
# prop/afqmc.py:132-134
avg_w        = mean(weights_new)
pop_shift_new = e_estimate - damping * log(avg_w) / dt
```
This damps the average weight toward 1, preventing population explosion or collapse.

---
## 6. Force Bias (Importance Sampling)

**File:** `meas/cisd.py`, function `force_bias_kernel_rw_rh`, lines 143–204

The force bias is the mixed estimator of $\hat{L}_\gamma = \sum_{ij} L_{\gamma,ij} \hat{a}^\dagger_i \hat{a}_j$:
$$F_\gamma = \frac{\langle\Psi_T | \hat{L}_\gamma | \phi\rangle}{\langle\Psi_T | \phi\rangle}$$

This decomposes into **HF + singles + doubles** contributions:

### 6.1 Setup
```python
green    = solve(wocc.T, walker.T)          # (nocc, norb)  ← Green's function G
green_occ = green[:, nocc:]                 # (nocc, nvir)  ← particle-hole sector x
greenp   = vstack([green_occ, -I_nvir])     # (norb, nvir)  ← "G'" matrix
```

### 6.2 HF contribution (fb_0)
$$F^{(0)}_\gamma = 2 \sum_{pj} L_{\gamma,pj}^{(\text{occ})} G_{pj}$$

```python
lg   = einsum("gpj,pj->g", rot_chol, green)   # rot_chol = L_γ[:nocc,:]
fb_0 = 2.0 * lg
```

### 6.3 Singles contribution (fb_1)
Two terms arise from the singles part of $\langle\Psi_T|$:

```python
ci1g    = einsum("ia,ia->", ci1, green_occ)          # scalar: c·x
ci1gp   = einsum("ia,it->pi", ci1, greenp)            # (nocc, norb)
gci1gp  = einsum("pj,pi->ij", green, ci1gp)           # (norb, norb)

fb_1_1  = 4.0 * ci1g * lg                            # diagonal HF-singles cross term
fb_1_2  = -2.0 * einsum("gij,ij->g", chol, gci1gp)   # off-diagonal matrix element
fb_1    = fb_1_1 + fb_1_2
```

### 6.4 Doubles contribution (fb_2)
```python
ci2g_c  = einsum("iajb,ia->jb", ci2, green_occ)       # Coulomb contraction
ci2g_e  = einsum("iajb,ib->ja", ci2, green_occ)       # Exchange contraction
# ... (combine into cisd_green tensor and contract with chol)
```

### 6.5 Normalized result
```python
overlap = 1.0 + 2.0*ci1g + 0.5*gci2g
return (fb_0 + fb_1 + fb_2) / overlap
```

> **Note:** The overlap denominator here uses a simplified form (`0.5*gci2g` vs `gci2g` in the full overlap) for numerical stability of the force bias.

---
## 7. Energy Evaluation (Mixed Estimator)

**File:** `meas/cisd.py`, function `energy_kernel_rw_rh`, lines 207–369

The mixed estimator energy is:
$$E = \frac{\langle\Psi_T | \hat{H} | \phi\rangle}{\langle\Psi_T | \phi\rangle} + h_0$$

The numerator decomposes into one-body (e1) and two-body (e2) parts, each with **HF + singles + doubles** contributions.

### 7.1 Green's function setup
Same as force bias: `green`, `green_occ`, `greenp` are computed first.

### 7.2 One-body energy (e1)

**HF part (e1_0):**
$$e_1^{(0)} = 2\sum_{pj} h_{pj}\, G_{pj}$$
```python
hg   = einsum("pj,pj->", h1[:nocc,:], green)
e1_0 = 2.0 * hg
```

**Singles correction (e1_1):** Two terms coupling $h_1$ matrix elements with CI-singles Green's function
```python
gpci1    = greenp @ ci1.T       # (norb, nocc)
ci1_green = gpci1 @ green       # (norb, norb)  ← matrix M such that e1_1 ~ h·M

e1_1_1 = 4.0 * ci1g * hg                              # HF × singles cross
e1_1_2 = -2.0 * einsum("ij,ij->", h1, ci1_green)      # singles matrix element
```

**Doubles correction (e1_2):** Similar structure with ci2-contracted Green's function tensors.

### 7.3 Two-body energy (e2)

**HF part (e2_0):** The standard Cholesky-factored 2-body energy
$$e_2^{(0)} = 2\left(\sum_\gamma l_\gamma^2\right) - \sum_\gamma \text{Tr}[L_\gamma^{(1)} (L_\gamma^{(1)})^T]$$
where $l_\gamma = \sum_{pj} L^{(\text{occ})}_{\gamma,pj}\, G_{pj}$ and $L^{(1)}_{\gamma,pq} = \sum_j L^{(\text{occ})}_{\gamma,pj}\, G_{qj}$.

```python
lg   = einsum("gpj,pj->g", rot_chol, green)         # (n_chol,)
lg1  = einsum("gpj,qj->gpq", rot_chol, green)        # (n_chol, nocc, nocc)

e2_0 = 2.0*(lg @ lg) - sum(lg1 * swapaxes(lg1,-1,-2))
```

**Singles correction (e2_1):** Three terms involving `lci1g` (Cholesky contracted with the singles correction matrix).

**Doubles correction (e2_2):**  Three terms. The last two involve $\sum_\gamma (L_\gamma G G') (c_2) (L_\gamma G G')$ — a quartic contraction that is the most expensive part.

Two memory modes are available:

| Mode | Implementation | File location |
|------|----------------|---------------|
| `"low"` | `lax.scan` over Cholesky vectors | `meas/cisd.py:298-333` |
| `"high"` | Full vectorization over all Choleskys at once | `meas/cisd.py:334-361` |

```python
# meas/cisd.py:298-333  (low-memory mode)
def scan_over_chol(carry, x):
    chol_i, rot_chol_i = x
    # ... compute e22 and e23 for this single Cholesky vector ...
    return (e22_acc + delta22, e23_acc + delta23), None

(e2_2_2_2, e2_2_3), _ = lax.scan(scan_over_chol, (zero, zero), (chol, rot_chol))
```

### 7.4 Final result
```python
overlap = 1.0 + 2.0*ci1g + gci2g
return (e1 + e2) / overlap + e0
```

---
## 8. Walker Management: Orthogonalization & Stochastic Reconfiguration

### 8.1 Walker Initialization
**File:** `walkers.py`, function `init_walkers`, lines 20–78

Walkers are initialized from the **natural orbitals** of the trial RDM1 (for CISD this is just the RHF RDM):
```python
# walkers.py:64-69  (restricted case)
dm_tot  = dm_up + dm_dn
natorbs = eigh(dm_tot)[1][:, ::-1][:, :nocc]   # largest-eigenvalue NOs
w0      = natorbs + 0j
walkers = broadcast_to(w0, (n_walkers, norb, nocc))  # all walkers start identical
```

---
### 8.2 Walker Orthogonalization (per block)
**File:** `walkers.py`, function `orthogonalize`, lines 151–171  
**File:** `prop/blocks.py`, line 76

After `n_prop_steps` of propagation, each walker is QR-decomposed to restore orthonormality while tracking the accumulated determinant (norm):
$$W = QR \implies |\phi\rangle \to Q, \quad \text{norm} = \prod_i R_{ii}$$

```python
# walkers.py:137-148
def _qr(mat):
    q, r = qr(mat, mode="reduced")
    phase = diag(r) / abs(diag(r))   # fix phase: make R diagonal real positive
    return q * conj(phase), prod(diag(phase[:,None]*r))

# prop/blocks.py:76
walkers_new = orthonormalize(state.walkers, sys.walker_kind)   # called after each block
```

---
### 8.3 Stochastic Reconfiguration
**File:** `walkers.py`, function `stochastic_reconfiguration`, lines 227–263  
**File:** `prop/blocks.py`, line 114

At the end of each block, walkers are **resampled** proportional to their weights (systematic resampling with a single uniform random number $\zeta$):

$$z_k = \frac{\sum_j |w_j|}{N}\,(k + \zeta), \quad k=0,\ldots,N-1$$
$$i_k = \min\left\{i : \sum_{j=0}^i |w_j| \geq z_k\right\}$$

Walker $i_k$ is copied into position $k$, and all new weights are set to $\bar{w} = \sum_j |w_j| / N$.

```python
# walkers.py:227-263
def stochastic_reconfiguration(w, weights, zeta, walker_kind, ...):
    cw   = cumsum(abs(weights))         # cumulative weights
    avg  = cw[-1] / n                   # target weight
    z    = cw[-1] * (arange(n) + zeta) / n
    idx  = searchsorted(cw, z)          # which walkers to copy
    return w[idx], full((n,), avg)      # resampled walkers + uniform weights

# prop/blocks.py:112-114
zeta   = random.uniform(subkey)
w_sr, weights_sr = sr_fn(state.walkers, state.weights, zeta, sys.walker_kind)
```

This prevents **weight collapse** (all weight on one walker) and **weight explosion** (one walker dominating due to fluctuations) without introducing bias.

---
## 9. Block-level Loop & Overall Driver

### 9.1 One QMC Block
**File:** `prop/blocks.py`, function `block`, lines 41–129

A single block consists of:
1. `n_prop_steps` AFQMC steps (via `lax.scan` for JIT efficiency)
2. Walker orthonormalization
3. Energy measurement (vmapped over all walkers)
4. Energy outlier rejection: $|E_k - E_{\text{ref}}| > \sqrt{2/\Delta\tau}$
5. Weighted average energy: $E_{\text{block}} = \sum_k w_k E_k / \sum_k w_k$
6. EMA update of `e_estimate`
7. Optional observable measurement (e.g., 1-RDM)
8. Stochastic reconfiguration

```python
# prop/blocks.py:70-128  (simplified)
state, _ = lax.scan(_scan_step, state, xs=None, length=n_prop_steps)   # 1. propagate
walkers  = orthonormalize(state.walkers, walker_kind)                   # 2. QR
e_samples = vmap(energy_kernel)(walkers, ham, meas_ctx, trial)          # 3. measure
e_block   = sum(w * e) / sum(w)                                         # 5. average
state.e_estimate = (1-α)*e_est + α*e_block                              # 6. EMA
walkers, weights = stochastic_reconfiguration(walkers, weights, zeta)   # 8. SR
```

---
### 9.2 Overall Driver
**File:** `driver.py`, function `run_qmc_energy`, lines 330–360

The top-level driver runs:
1. **Initialization** (`prop/afqmc.py:init_prop_state`): walkers from trial NOs, initial overlaps + energy estimate
2. **Build contexts** (`_build_prop_ctx`, `build_meas_ctx`): precompute `exp_h1_half`, `mf_shifts`, `rot_chol`, `lci1`
3. **Equilibration**: `n_eql_blocks` blocks without collecting statistics
4. **Sampling**: `n_blocks` blocks, collect `(energy, weight)` per block
5. **Statistical analysis** (`stat_utils.py`): blocking analysis to estimate error bar

```python
# driver.py (schematic)
state    = init_prop_state(sys, ham, trial_ops, trial_data, meas_ops, params)
prop_ctx = build_prop_ctx(ham, rdm1, params)
meas_ctx = build_meas_ctx(ham, trial_data)

for _ in range(n_eql_blocks):
    state, _ = block(state, ...)           # equilibration

energies, weights = [], []
for _ in range(n_blocks):
    state, obs = block(state, ...)         # sampling
    energies.append(obs.scalars["energy"])
    weights.append(obs.scalars["weight"])

mean, err = blocking_analysis(energies, weights)
```

---
### 9.3 User-facing API
**File:** `afqmc.py`, class `Afqmc`, lines 74–298

```python
from pyscf import gto, scf, cc
from ad_afqmc_prototype import Afqmc

mol  = gto.M(atom="H 0 0 0; H 0 0 1.4", basis="sto-3g")
mf   = scf.RHF(mol).run()
mycc = cc.CCSD(mf).run()        # provides ci1, ci2 via t1, t2 amplitudes

afqmc = Afqmc(mycc)             # wraps PySCF object
afqmc.n_walkers    = 100
afqmc.n_eql_blocks = 10
afqmc.n_blocks     = 100
e_mean, e_err = afqmc.kernel()  # → driver.run_qmc_energy()
```

---
## 10. Call Graph Summary

```
Afqmc.kernel()                              afqmc.py
└── driver.run_qmc_energy()
    ├── init_prop_state()                   prop/afqmc.py:19
    │   ├── init_walkers(sys, rdm1, nw)     walkers.py:20
    │   └── vmap(meas_ops.overlap)          trial/cisd.py → overlap_r
    ├── _build_prop_ctx(ham, rdm1, dt)      prop/chol_afqmc_ops.py:103
    │   ├── _mf_shifts(ham, rdm1)           → mf_shifts
    │   ├── _get_h1_eff(ham, mf)            → h1_eff
    │   └── expm(-dt/2 * h1_eff)            → exp_h1_half
    ├── build_meas_ctx(ham, trial)          meas/cisd.py:72
    │   └── precompute rot_chol, lci1
    │
    ├── [equilibration loop]
    │   └── block(state, ...)               prop/blocks.py:41
    │       ├── lax.scan( afqmc_step() )    prop/afqmc.py:79  (×n_prop_steps)
    │       │   ├── vmap(force_bias_kernel) meas/cisd.py:143
    │       │   ├── vmap(apply_trotter)     prop/chol_afqmc_ops.py:198
    │       │   │   ├── exp_h1_half @ w
    │       │   │   ├── taylor_expm_action  prop/utils.py:20
    │       │   │   └── exp_h1_half @ w
    │       │   ├── vmap(overlap_r)         trial/cisd.py:62
    │       │   └── phaseless weight update
    │       ├── orthonormalize(walkers)     walkers.py:174
    │       ├── vmap(energy_kernel)         meas/cisd.py:207
    │       ├── e_estimate EMA update
    │       └── stochastic_reconfiguration  walkers.py:227
    │
    └── [sampling loop]  (same as equilibration, plus data collection)
        └── blocking_analysis()            stat_utils.py
```

---
## Summary Table

| Component | File | Function | Lines |
|-----------|------|----------|-------|
| Trial data (CISD) | `trial/cisd.py` | `CisdTrial` | 13–51 |
| Trial RDM1 | `trial/cisd.py` | `get_rdm1` | 54–59 |
| **Overlap** | `trial/cisd.py` | `overlap_r` | 62–76 |
| Hamiltonian | `ham/chol.py` | `HamChol` | 12–45 |
| Measurement context | `meas/cisd.py` | `build_meas_ctx` | 72–90 |
| **Force bias** | `meas/cisd.py` | `force_bias_kernel_rw_rh` | 143–204 |
| **Energy** | `meas/cisd.py` | `energy_kernel_rw_rh` | 207–369 |
| 1-RDM observable | `meas/cisd.py` | `rdm1_kernel_rw` | 372–421 |
| Walker initialization | `walkers.py` | `init_walkers` | 20–78 |
| Walker orthogonalization | `walkers.py` | `orthogonalize` | 151–171 |
| **Stochastic reconfiguration** | `walkers.py` | `stochastic_reconfiguration` | 227–263 |
| Propagation context | `prop/chol_afqmc_ops.py` | `_build_prop_ctx` | 103–127 |
| **Trotter step** | `prop/chol_afqmc_ops.py` | `_apply_trotter_r` | 198–208 |
| Taylor exp | `prop/utils.py` | `taylor_expm_action` | 20–29 |
| **Phaseless step** | `prop/afqmc.py` | `afqmc_step` | 79–144 |
| Prop state init | `prop/afqmc.py` | `init_prop_state` | 19–76 |
| **Block loop** | `prop/blocks.py` | `block` | 41–129 |
| **QMC driver** | `driver.py` | `run_qmc_energy` | 330–360 |
| User API | `afqmc.py` | `Afqmc.kernel` | 276–298 |